# S2.4 — Lazy Evaluation (✅ "Once we call an ACTION then lazy man starts working")
**Date completed:** May 2025  
**Status:** In Progress  
**Interview covered:** Q10 — What is lazy evaluation in Spark?

# Prove lazy evaluation

In [0]:
import time

print("=== PROVING LAZY EVALUATION ===")
print()

# Step 1 — Transformations (should be instant — no execution)
print("Building DAG - Transformation only.....")
start = time.time()

df = spark.range(0, 10000000)            # 10 million rows
df = df.filter(df.id > 5000000)          # 5 million rows
df = df.filter(df.id % 2 == 0)           # filter even numbers only

end = time.time()
print(f"DAG built in: {round(end - start, 4)} seconds")
print("No data processed yet — Spark just built the plan")
print()

# Step 2 — Action (this triggers actual execution)
print("Calling ACTION — now Spark executes...")
start = time.time()

result = df.count()

end = time.time()

print(f"Action completed in: {round(end - start, 4)} seconds")
print(f"Result: {result} rows")
print()
print("DAG built instantly. Execution happened ONLY at count()")

In [0]:
# Total passes through data = 4 separate executions
# Total rows processed = 10M + 10M + 5M + 2.5M = 27.5M rows

df = spark.range(0, 10000000)    # EXECUTES → processes 10M rows (Pass 1)
df = df.filter(df.id > 5000000)  # EXECUTES → processes 10M rows (Pass 2)
df = df.filter(df.id % 2 == 0)   # EXECUTES → processes 5M rows  (Pass 3)
result = df.count()              # EXECUTES → processes 2.5M rows (Pass 4)

In [0]:
# Total passes through data = 1 execution
# Catalyst applies BOTH filters in one pass
# Total rows processed = 10M only (once)

df = spark.range(0, 10000000)    # No execution
df = df.filter(df.id > 5000000)  # No execution
df = df.filter(df.id % 2 == 0)   # No execution
result = df.count()              # ONE execution — all steps combined

- Eager  = 27.5M rows processed
- Lazy   = 10M rows processed
- Saving = 63% less work

In [0]:
df = spark.range(0, 10000000)   # Lazy → adds Node 1 to DAG
df = df.filter(df.id > 5000000) # Lazy → adds Node 2 to DAG  
df = df.filter(df.id % 2 == 0)  # Lazy → adds Node 3 to DAG
                                 # DAG now has 3 nodes — no execution yet

result = df.count()              # Action → Catalyst optimizes DAG
                                 # Then executes optimized DAG
                                 # Lazy man wakes up

- Lazy man collecting information = LAZY EVALUATION
- The collection of information   = DAG
- Boss says execute now            = ACTION

## Key Takeaways — Lazy Evaluation

## Proof from code
- DAG built in: 0.06 seconds (no data processed)
- Action executed in: 0.36 seconds (10M rows processed)
- Lazy = plan first, execute only when forced

## DAG and Lazy Evaluation — how they work together
- Lazy Evaluation = the behaviour (don't execute until forced)
- DAG = the plan built BECAUSE of lazy behaviour
- Action = triggers Catalyst to optimize DAG, then execute

## Eager vs Lazy comparison
| | Eager (Python) | Lazy (Spark) |
|--|----------------|-------------|
| Executes | Immediately | Only on action |
| Passes through data | 4 times | 1 time |
| Rows processed | 27.5M | 10M |
| Efficiency | Low | High |

## Interview Answer — Q10
"Spark is lazy — transformations build a DAG without executing.
When an action is called, Catalyst optimizes the full DAG first,
then executes in one efficient pass. This reduces data processing
by combining all operations."